# Bus Delay Prediction - Data Preparation Pipeline
## Machine Learning-Based Prediction of Bus Delay Duration in Scotland's Central Belt

This notebook documents the complete data preparation journey from raw data inspection
through to producing a final synthetic delay dataset ready for ML modelling.

**Project Focus:** Edinburgh, Glasgow, and Paisley bus services

**Data Sources:**
- Timetable data: GTFS timetables from BODS (Bus Open Data Service) archive, 21 daily parquet files (Jan 1-21, 2026)
- Weather data: Visual Crossing hourly weather data, 21 daily parquet files (Jan 1-21, 2026)

**Why Synthetic Delays?**
Real-time arrival data for Scotland's Central Belt is not available through BODS.
Scottish operators (Lothian Buses, First Glasgow, McGill's) do not feed real-time data into
the national open data infrastructure. This was verified by inspecting GTFS-RT feeds (see Stage 1).
We therefore use real timetable structures with research-calibrated synthetic delays.

---

## Stage 1: Inspect the Timetable Data Structure

We begin by loading one day's timetable (tim1.parquet) to understand the data structure.
Each parquet file contains the full UK bus timetable for that day - a pre-joined table of
GTFS stop_times and trips data. Each row represents one **stop visit**: a specific bus trip
arriving at a specific stop.

In [ ]:
import pandas as pd
import numpy as np

# Load day 1 timetable to inspect its structure
tim = pd.read_parquet("timetables/tim1.parquet")

print("=== TIMETABLE STRUCTURE ===")
print(f"Shape: {tim.shape}")
print(f"Columns: {list(tim.columns)}")
print(f"\nFirst 5 rows:")
tim.head(5)

### Key Observations:
- Each file has ~8.97 million rows covering ALL UK bus services
- Key columns: `trip_id`, `arrival_time`, `departure_time`, `stop_id`, `stop_sequence`, `route_id`
- `stop_id` uses NaPTAN codes where the first 3 digits indicate the geographic area
- The data is already a join of GTFS stop_times + trips tables

### Understanding stop_id Prefixes (NaPTAN ATCO Codes)

UK bus stops use the NaPTAN (National Public Transport Access Nodes) system.
The first 3 digits of stop_id map to geographic regions:
- **620** = Edinburgh & Lothians
- **609** = Glasgow & surrounding areas  
- **608** = Renfrewshire (including Paisley)

Let's verify which prefixes are present and how many rows belong to our target cities.

In [ ]:
# Check stop_id prefix distribution across the full UK dataset
tim['prefix'] = tim['stop_id'].astype(str).str[:3]

print("Top 20 stop_id prefixes (all UK):")
print(tim['prefix'].value_counts().head(20))

# Filter to our three target cities
focus_prefixes = ['620', '609', '608']
focus = tim[tim['prefix'].isin(focus_prefixes)]

print(f"\n=== FILTERING TO EDINBURGH, GLASGOW, PAISLEY ===")
print(f"Edinburgh (620): {tim[tim['prefix']=='620'].shape[0]:,} rows")
print(f"Glasgow (609):   {tim[tim['prefix']=='609'].shape[0]:,} rows")
print(f"Paisley (608):   {tim[tim['prefix']=='608'].shape[0]:,} rows")
print(f"\nCombined: {focus.shape[0]:,} rows out of {tim.shape[0]:,} total UK rows")
print(f"Unique trips: {focus['trip_id'].nunique():,}")
print(f"Unique routes: {focus['route_id'].nunique()}")
print(f"Unique stops: {focus['stop_id'].nunique():,}")

del tim  # Free memory

### Result:
- ~3.93 million rows per day for our three cities (out of ~8.97M total UK)
- 96,231 unique trips, 446 routes, 5,502 stops
- Edinburgh dominates with ~2.2M rows, followed by Glasgow (~1.5M), then Paisley (~197K)

---

## Stage 2: Route Selection - Choosing the Top Routes

3.93 million rows per day × 21 days = ~82 million rows. This is too large for practical
ML processing. We select the **highest-frequency routes** from each city, which are the
most operationally significant and provide the richest data for pattern detection.

We take the **top 15 routes from Edinburgh, top 15 from Glasgow, and top 5 from Paisley**
(proportional to each city's share of the data). This ensures balanced representation
across all three cities rather than being dominated by Edinburgh.

In [ ]:
# Reload day 1 and filter to our cities
tim = pd.read_parquet("timetables/tim1.parquet")
tim['prefix'] = tim['stop_id'].astype(str).str[:3]
focus_prefixes = ['620', '609', '608']
focus = tim[tim['prefix'].isin(focus_prefixes)]

# Split by city
edin = focus[focus['prefix'] == '620']
glas = focus[focus['prefix'] == '609']
pais = focus[focus['prefix'] == '608']

# Get top routes per city ranked by number of unique trips
top_edin = edin.groupby('route_id')['trip_id'].nunique().sort_values(ascending=False).head(15).index.tolist()
top_glas = glas.groupby('route_id')['trip_id'].nunique().sort_values(ascending=False).head(15).index.tolist()
top_pais = pais.groupby('route_id')['trip_id'].nunique().sort_values(ascending=False).head(5).index.tolist()

# Combine (removing any duplicates if a route serves multiple cities)
all_selected = list(set(top_edin + top_glas + top_pais))
selected_data = focus[focus['route_id'].isin(all_selected)]

print(f"Selected {len(all_selected)} unique routes")
print(f"Total rows (1 day): {selected_data.shape[0]:,}")
print(f"Unique trips: {selected_data['trip_id'].nunique():,}")
print(f"Unique stops: {selected_data['stop_id'].nunique():,}")
print(f"Estimated 21 days: ~{selected_data.shape[0] * 21:,} rows")

# Show trip characteristics
stops_per_trip = selected_data.groupby('trip_id')['stop_id'].count()
print(f"\nAvg stops per trip: {stops_per_trip.mean():.1f}")
print(f"Median stops per trip: {stops_per_trip.median():.1f}")

# Show the selected routes
print(f"\n--- Top 15 Edinburgh Routes ---")
for r in top_edin:
    rdata = edin[edin['route_id'] == r]
    headsigns = rdata['trip_headsign'].dropna().unique()[:3]
    print(f"  Route {r}: {rdata['trip_id'].nunique()} trips, headsigns={list(headsigns)}")

print(f"\n--- Top 15 Glasgow Routes ---")
for r in top_glas:
    rdata = glas[glas['route_id'] == r]
    headsigns = rdata['trip_headsign'].dropna().unique()[:3]
    print(f"  Route {r}: {rdata['trip_id'].nunique()} trips, headsigns={list(headsigns)}")

print(f"\n--- Top 5 Paisley Routes ---")
for r in top_pais:
    rdata = pais[pais['route_id'] == r]
    headsigns = rdata['trip_headsign'].dropna().unique()[:3]
    print(f"  Route {r}: {rdata['trip_id'].nunique()} trips, headsigns={list(headsigns)}")

del tim, focus, edin, glas, pais, selected_data  # Free memory

### Result:
- 32 selected routes covering Edinburgh (15), Glasgow (15), Paisley (5)
- ~1.41 million rows per day, ~29.7 million estimated over 21 days
- Mix of urban city services, cross-city services, and airport routes
- Average ~40 stops per trip

---

## Stage 3: Inspect the Weather Data

Weather data was obtained from Visual Crossing for the same 21-day period.
Each daily parquet file contains hourly observations across Scottish regions.
We need to verify the structure, available regions, and map them to our cities.

In [ ]:
# Load day 1 weather to inspect structure
wea = pd.read_parquet("weather/wea1.parquet")

print("=== WEATHER DATA STRUCTURE ===")
print(f"Shape: {wea.shape}")
print(f"Columns: {list(wea.columns)}")
print(f"\nRegions available: {wea['region'].unique()}")
print(f"Rows per region:")
print(wea['region'].value_counts())
print(f"\nDate/time range: {wea['datetime'].min()} to {wea['datetime'].max()}")
print(f"Precip range: {wea['precip'].min()} to {wea['precip'].max()}")
print(f"Temp range: {wea['temp'].min()} to {wea['temp'].max()}")
print(f"Conditions: {wea['conditions'].unique()}")
print(f"Visibility column null count: {wea['visibility'].isna().sum()} out of {wea.shape[0]}")

print(f"\nSample Central region data:")
print(wea[wea['region']=='Central'][['datetime','temp','precip','conditions','windspeed']].head())

print(f"\nSample East region data:")
print(wea[wea['region']=='East'][['datetime','temp','precip','conditions','windspeed']].head())

### Key Observations:
- 5 regions: Central, East, North, Northeast, South
- 18 hours per region (06:00 to 23:00) = 90 rows per day
- **Visibility column is empty** (all NaN) - will be dropped
- Good variety of conditions: clear, overcast, snow, drizzle, rain
- Useful features: temp, precip, preciptype, conditions, windspeed, humidity, cloudcover

### Weather Region to City Mapping:
- **Edinburgh** (stop prefix 620) → **East** weather region
- **Glasgow** (stop prefix 609) → **Central** weather region  
- **Paisley** (stop prefix 608) → **Central** weather region

---

## Stage 4: Test Merge - Timetable + Weather (Day 1)

Before processing all 21 days, we test the merge logic on day 1.
Each timetable row has an `arrival_time` with an hour component.
Each weather row has a `datetime` with an hour component.
We merge by matching the **hour** and **weather region** so every stop visit
gets the weather conditions prevailing in its city at that time.

In [ ]:
# Load day 1 data
tim = pd.read_parquet("timetables/tim1.parquet")
wea = pd.read_parquet("weather/wea1.parquet")

# Filter timetable to our 3 cities and selected routes
tim['prefix'] = tim['stop_id'].astype(str).str[:3]
focus_prefixes = ['620', '609', '608']
focus = tim[tim['prefix'].isin(focus_prefixes)]

# Re-derive top routes (same logic as Stage 2)
edin = focus[focus['prefix'] == '620']
glas = focus[focus['prefix'] == '609']
pais = focus[focus['prefix'] == '608']
top_edin = edin.groupby('route_id')['trip_id'].nunique().sort_values(ascending=False).head(15).index.tolist()
top_glas = glas.groupby('route_id')['trip_id'].nunique().sort_values(ascending=False).head(15).index.tolist()
top_pais = pais.groupby('route_id')['trip_id'].nunique().sort_values(ascending=False).head(5).index.tolist()
all_selected = list(set(top_edin + top_glas + top_pais))

filtered = focus[focus['route_id'].isin(all_selected)].copy()

# Add city and weather region columns
filtered['city'] = filtered['prefix'].map({'620': 'Edinburgh', '609': 'Glasgow', '608': 'Paisley'})
filtered['weather_region'] = filtered['prefix'].map({'620': 'East', '609': 'Central', '608': 'Central'})

# Extract hour from arrival_time for merging
filtered['hour'] = filtered['arrival_time'].str.split(':').str[0].astype(int)

# Prepare weather: keep only Central and East regions, extract hour
wea = wea[wea['region'].isin(['Central', 'East'])].copy()
wea['hour'] = pd.to_datetime(wea['datetime']).dt.hour

# Merge timetable with weather on hour and region
merged = filtered.merge(
    wea,
    left_on=['weather_region', 'hour'],
    right_on=['region', 'hour'],
    how='left'
)

print(f"Filtered timetable rows: {filtered.shape[0]:,}")
print(f"After merge with weather: {merged.shape[0]:,}")
print(f"Rows with missing weather (outside 06:00-23:00): {merged['temp'].isna().sum():,}")
print(f"\nMerged columns: {list(merged.columns)}")
print(f"\nSample merged row (8am Edinburgh):")
print(merged[merged['hour']==8].head(1).to_string())

del tim, wea, focus, edin, glas, pais, filtered, merged  # Free memory

### Result:
- Merge works correctly - 1,412,856 rows maintained
- ~51,526 rows have missing weather (trips before 06:00 or after 23:00)
- These will be filled with the nearest available weather observation
- Each stop visit now has its corresponding hourly weather conditions

---

## Stage 5: Delay Synthesis - Methodology

Since real arrival time data is unavailable for Scotland, we generate realistic
delay values using patterns established in published transport research.

### Synthesis Components:

**1. Base Delay per Trip** - Each trip gets a starting delay drawn from a normal
distribution (mean=1.5 min, std=2.5 min). This allows both early and late buses.
Bus delays consistently follow right-skewed distributions in empirical studies
(Mazloumi et al., 2010; Cats et al., 2014).

**2. Time-of-Day Effect** - Delay is multiplied by a factor based on the hour:
- Rush hours (7-9 AM, 4-6 PM): multiplier 1.3-1.5x
- Midday: multiplier 0.8-0.9x
- Late night: multiplier 0.4x

**3. Weather Effect** - Additional delay based on conditions:
- Heavy precipitation: up to +3 minutes
- Snow: +1 to +3 minutes
- Rain/drizzle: +0.3 to +1.2 minutes
- High wind (>40 km/h): +0.3 to +1.0 minutes
- Freezing temperatures (≤0°C): +0.3 to +1.0 minutes
- Based on Stover & McCormack (2012) and Wei & Chen (2019)

**4. Day-of-Week Effect** - Weekdays get higher delays than weekends:
- Monday/Friday: multiplier 1.05x
- Tuesday-Thursday: multiplier 1.0x
- Saturday: multiplier 0.7x
- Sunday: multiplier 0.5x

**5. Delay Propagation Along Route** - Delays accumulate: a bus late at stop 5
is likely at least as late at stop 15. Small random increments (-0.05 to +0.12 min)
are added at each stop, with occasional recovery.

**6. Random Noise** - Small random component (±0.3 min) for natural variation.

### Calibration:
Parameters were tuned on day 1 to produce realistic distributions:
- Mean delay: ~2.6 min (target: 2-4 min from UK punctuality studies)
- ~15% early, ~52% on time, ~17% moderately late, ~1% very late
- Clear rush hour peaks and weather effects visible

---

## Stage 6: Test Synthesis on Day 1

Before running the full 21-day pipeline, we test the complete synthesis on day 1
to verify the delay distribution is realistic.

In [ ]:
import pandas as pd
import numpy as np

# ---- Load and filter day 1 ----
tim = pd.read_parquet("timetables/tim1.parquet")
tim['prefix'] = tim['stop_id'].astype(str).str[:3]
focus_prefixes = ['620', '609', '608']
focus = tim[tim['prefix'].isin(focus_prefixes)]

edin = focus[focus['prefix'] == '620']
glas = focus[focus['prefix'] == '609']
pais = focus[focus['prefix'] == '608']
top_edin = edin.groupby('route_id')['trip_id'].nunique().sort_values(ascending=False).head(15).index.tolist()
top_glas = glas.groupby('route_id')['trip_id'].nunique().sort_values(ascending=False).head(15).index.tolist()
top_pais = pais.groupby('route_id')['trip_id'].nunique().sort_values(ascending=False).head(5).index.tolist()
all_selected = list(set(top_edin + top_glas + top_pais))
filtered = focus[focus['route_id'].isin(all_selected)].copy()

filtered['city'] = filtered['prefix'].map({'620': 'Edinburgh', '609': 'Glasgow', '608': 'Paisley'})
filtered['weather_region'] = filtered['prefix'].map({'620': 'East', '609': 'Central', '608': 'Central'})
filtered['hour'] = filtered['arrival_time'].str.split(':').str[0].astype(int)

wea = pd.read_parquet("weather/wea1.parquet")
wea = wea[wea['region'].isin(['Central', 'East'])].copy()
wea['hour'] = pd.to_datetime(wea['datetime']).dt.hour

merged = filtered.merge(wea, left_on=['weather_region', 'hour'], right_on=['region', 'hour'], how='left')

# Fill missing weather for early/late hours
for col in ['temp', 'precip', 'windspeed', 'humidity', 'cloudcover']:
    merged[col] = merged.groupby('weather_region')[col].transform(lambda x: x.bfill().ffill())
merged['conditions'] = merged['conditions'].fillna('Clear sky')
merged['preciptype'] = merged['preciptype'].fillna('None')

merged['date'] = '2026-01-01'
merged['day_of_week'] = 2  # Wednesday

# ---- SYNTHESIS ----
np.random.seed(42)

# Time-of-day multipliers
hour_multiplier = {
    0: 0.4, 1: 0.4, 2: 0.4, 3: 0.4, 4: 0.5, 5: 0.6,
    6: 0.8, 7: 1.3, 8: 1.5, 9: 1.2, 10: 0.9, 11: 0.8,
    12: 0.85, 13: 0.85, 14: 0.9, 15: 1.1, 16: 1.4, 17: 1.5,
    18: 1.2, 19: 0.9, 20: 0.7, 21: 0.5, 22: 0.4, 23: 0.4
}

# Day-of-week multipliers
dow_multiplier = {0: 1.05, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.05, 5: 0.7, 6: 0.5}

# Weather delay function
def weather_delay(row):
    extra = 0.0
    if row['precip'] > 0:
        extra += min(row['precip'] * 1.5, 3.0)
    cond = str(row['conditions']).lower()
    if 'snow' in cond:
        extra += np.random.uniform(1.0, 3.0)
    elif 'rain' in cond or 'drizzle' in cond:
        extra += np.random.uniform(0.3, 1.2)
    if pd.notna(row['windspeed']) and row['windspeed'] > 40:
        extra += np.random.uniform(0.3, 1.0)
    if pd.notna(row['temp']) and row['temp'] <= 0:
        extra += np.random.uniform(0.3, 1.0)
    return extra

# Apply synthesis
unique_trips = merged['trip_id'].unique()
trip_base = pd.Series(np.random.normal(loc=1.5, scale=2.5, size=len(unique_trips)), index=unique_trips)

print("Calculating weather delays...")
merged['weather_extra'] = merged.apply(weather_delay, axis=1)
merged['base_delay'] = merged['trip_id'].map(trip_base)
merged['hour_mult'] = merged['hour'].map(hour_multiplier)
merged['dow_mult'] = merged['day_of_week'].map(dow_multiplier)

merged['pre_prop_delay'] = (merged['base_delay'] * merged['hour_mult'] * merged['dow_mult']) + merged['weather_extra']

# Propagation along route
merged = merged.sort_values(['trip_id', 'stop_sequence'])
merged['stop_increment'] = np.random.uniform(-0.05, 0.12, size=len(merged))
merged['cumulative_increment'] = merged.groupby('trip_id')['stop_increment'].cumsum()

merged['delay_minutes'] = (
    merged['pre_prop_delay'] + merged['cumulative_increment'] + np.random.normal(0, 0.3, size=len(merged))
).clip(-3, 30).round(1)

# ---- VERIFICATION ----
print(f"\n=== DAY 1 SYNTHESIS VERIFICATION ===")
print(f"Total rows: {merged.shape[0]:,}")
print(f"Mean delay: {merged['delay_minutes'].mean():.2f} min")
print(f"Median delay: {merged['delay_minutes'].median():.2f} min")
early = (merged['delay_minutes'] < 0).mean() * 100
ontime = ((merged['delay_minutes'] >= -1) & (merged['delay_minutes'] <= 3)).mean() * 100
late = (merged['delay_minutes'] > 5).mean() * 100
vlate = (merged['delay_minutes'] > 10).mean() * 100
print(f"% early (<0): {early:.1f}%")
print(f"% on time (-1 to 3): {ontime:.1f}%")
print(f"% late (>5): {late:.1f}%")
print(f"% very late (>10): {vlate:.1f}%")

print(f"\nDelay by city:")
print(merged.groupby('city')['delay_minutes'].agg(['mean','median']).round(2).to_string())

print(f"\nDelay by hour (mean):")
hourly = merged.groupby('hour')['delay_minutes'].mean()
for h in sorted(hourly.index):
    if pd.notna(hourly[h]):
        bar = '#' * max(0, int(hourly[h] * 3))
        print(f"  {h:02d}:00  {hourly[h]:5.2f} min  {bar}")

del tim, focus, edin, glas, pais, filtered, merged  # Free memory

### Verification Result:
- Mean delay ~2.6 min — within 2-4 min range from UK bus punctuality literature
- ~15% early, ~52% on time, ~16% late, ~1% very late — realistic distribution
- Rush hour peaks at 08:00 and 17:00 clearly visible
- City differences present (Paisley slightly higher delays)
- **Calibration confirmed — ready for full 21-day run**

---

## Stage 7: Full 21-Day Production Pipeline

Now we run the complete pipeline across all 21 days.

**Memory management:** With 8GB RAM, we cannot load all days at once.
We process one day at a time, save to temporary parquet files,
then combine at the end.

**Route consistency:** The top routes are determined from day 1 and applied
to all days, ensuring consistent route selection across the dataset.

In [ ]:
import pandas as pd
import numpy as np
import os
import gc
from datetime import datetime, timedelta

# ===============================================================
# CONFIGURATION
# ===============================================================
TIMETABLE_DIR = "timetables"
WEATHER_DIR = "weather"
OUTPUT_FILE = "scotland_bus_delays_synthetic.parquet"
NUM_DAYS = 21
START_DATE = datetime(2026, 1, 1)

# City filters based on NaPTAN ATCO stop_id prefixes
FOCUS_PREFIXES = ['620', '609', '608']
CB_PREFIX_TO_CITY = {'620': 'Edinburgh', '609': 'Glasgow', '608': 'Paisley'}
CB_PREFIX_TO_WEATHER = {'620': 'East', '609': 'Central', '608': 'Central'}

# Time-of-day delay multipliers (rush hour = higher)
HOUR_MULTIPLIER = {
    0: 0.4, 1: 0.4, 2: 0.4, 3: 0.4, 4: 0.5, 5: 0.6,
    6: 0.8, 7: 1.3, 8: 1.5, 9: 1.2, 10: 0.9, 11: 0.8,
    12: 0.85, 13: 0.85, 14: 0.9, 15: 1.1, 16: 1.4, 17: 1.5,
    18: 1.2, 19: 0.9, 20: 0.7, 21: 0.5, 22: 0.4, 23: 0.4
}

# Day-of-week delay multipliers (weekends = lower)
DOW_MULTIPLIER = {0: 1.05, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.05, 5: 0.7, 6: 0.5}

np.random.seed(42)

# ===============================================================
# STEP 1: Determine top routes from day 1
# We fix the route selection from day 1 and apply it to all days
# to ensure consistency across the dataset.
# ===============================================================
print("Step 1: Identifying top routes from day 1...")
tim1 = pd.read_parquet(os.path.join(TIMETABLE_DIR, "tim1.parquet"))
tim1['prefix'] = tim1['stop_id'].astype(str).str[:3]
focus1 = tim1[tim1['prefix'].isin(FOCUS_PREFIXES)]

edin1 = focus1[focus1['prefix'] == '620']
glas1 = focus1[focus1['prefix'] == '609']
pais1 = focus1[focus1['prefix'] == '608']

top_edin = edin1.groupby('route_id')['trip_id'].nunique().sort_values(ascending=False).head(15).index.tolist()
top_glas = glas1.groupby('route_id')['trip_id'].nunique().sort_values(ascending=False).head(15).index.tolist()
top_pais = pais1.groupby('route_id')['trip_id'].nunique().sort_values(ascending=False).head(5).index.tolist()
SELECTED_ROUTES = list(set(top_edin + top_glas + top_pais))

print(f"  Selected {len(SELECTED_ROUTES)} routes: Edinburgh(15), Glasgow(15), Paisley(5)")
del tim1, focus1, edin1, glas1, pais1
gc.collect()

# ===============================================================
# HELPER FUNCTION: Calculate weather-based delay
# Based on Stover & McCormack (2012) and Wei & Chen (2019)
# ===============================================================
def weather_delay(precip, conditions, windspeed, temp):
    """Calculate additional delay minutes caused by weather conditions."""
    extra = 0.0
    # Precipitation intensity effect
    if precip > 0:
        extra += min(precip * 1.5, 3.0)
    # Condition-specific effects
    cond = str(conditions).lower()
    if 'snow' in cond:
        extra += np.random.uniform(1.0, 3.0)
    elif 'rain' in cond or 'drizzle' in cond:
        extra += np.random.uniform(0.3, 1.2)
    # High wind effect
    if pd.notna(windspeed) and windspeed > 40:
        extra += np.random.uniform(0.3, 1.0)
    # Freezing/icy conditions
    if pd.notna(temp) and temp <= 0:
        extra += np.random.uniform(0.3, 1.0)
    return extra

# ===============================================================
# HELPER FUNCTION: Process one day end-to-end
# Loads timetable + weather, filters, merges, synthesizes delays
# ===============================================================
def process_one_day(day_num):
    """Process a single day: filter, merge weather, synthesize delays."""
    current_date = START_DATE + timedelta(days=day_num - 1)
    date_str = current_date.strftime('%Y-%m-%d')
    dow = current_date.weekday()  # 0=Monday ... 6=Sunday

    # --- Load and filter timetable ---
    tim = pd.read_parquet(os.path.join(TIMETABLE_DIR, f"tim{day_num}.parquet"))
    tim['prefix'] = tim['stop_id'].astype(str).str[:3]
    filtered = tim[
        tim['prefix'].isin(FOCUS_PREFIXES) & tim['route_id'].isin(SELECTED_ROUTES)
    ].copy()
    del tim

    if filtered.shape[0] == 0:
        print(f"  Day {day_num}: No matching rows, skipping")
        return None

    # --- Add city and weather region columns ---
    filtered['city'] = filtered['prefix'].map(CB_PREFIX_TO_CITY)
    filtered['weather_region'] = filtered['prefix'].map(CB_PREFIX_TO_WEATHER)
    filtered['hour'] = filtered['arrival_time'].str.split(':').str[0].astype(int)
    # GTFS allows hours >23 for trips past midnight (e.g. 25:30 = 1:30am next day)
    # Cap at 23 for weather matching purposes
    filtered['weather_hour'] = filtered['hour'].clip(upper=23)

    # --- Load and prepare weather ---
    wea = pd.read_parquet(os.path.join(WEATHER_DIR, f"wea{day_num}.parquet"))
    wea = wea[wea['region'].isin(['Central', 'East'])].copy()
    wea['hour'] = pd.to_datetime(wea['datetime']).dt.hour
    wea = wea.drop(columns=['visibility'], errors='ignore')  # Always empty

    # --- Merge timetable with weather ---
    filtered = filtered.merge(
        wea,
        left_on=['weather_region', 'weather_hour'],
        right_on=['region', 'hour'],
        how='left',
        suffixes=('', '_wea')
    )

    # Fill missing weather (trips before 06:00 or after 23:00)
    for col in ['temp', 'precip', 'windspeed', 'humidity', 'cloudcover']:
        filtered[col] = filtered.groupby('weather_region')[col].transform(
            lambda x: x.bfill().ffill()
        )
    filtered['conditions'] = filtered['conditions'].fillna('Clear sky')
    filtered['preciptype'] = filtered['preciptype'].fillna('None')

    # --- Add date features ---
    filtered['date'] = date_str
    filtered['day_of_week'] = dow
    filtered['is_weekend'] = 1 if dow >= 5 else 0

    # --- SYNTHESIZE DELAYS ---
    n_rows = filtered.shape[0]
    unique_trips = filtered['trip_id'].unique()

    # 1. Base delay per trip: normal distribution (mean=1.5, std=2.5)
    trip_base = pd.Series(
        np.random.normal(loc=1.5, scale=2.5, size=len(unique_trips)),
        index=unique_trips
    )

    # 2. Weather delay: vectorized loop over all rows
    weather_extra = np.zeros(n_rows)
    precip_vals = filtered['precip'].values
    cond_vals = filtered['conditions'].values
    wind_vals = filtered['windspeed'].values
    temp_vals = filtered['temp'].values
    for i in range(n_rows):
        weather_extra[i] = weather_delay(precip_vals[i], cond_vals[i], wind_vals[i], temp_vals[i])

    # 3. Combine: base × time_multiplier × dow_multiplier + weather
    filtered['base_delay'] = filtered['trip_id'].map(trip_base)
    filtered['hour_mult'] = filtered['hour'].map(HOUR_MULTIPLIER).fillna(0.4)
    filtered['dow_mult'] = filtered['day_of_week'].map(DOW_MULTIPLIER)

    filtered['pre_prop_delay'] = (
        filtered['base_delay'] * filtered['hour_mult'] * filtered['dow_mult']
    ) + weather_extra

    # 4. Delay propagation along route
    filtered = filtered.sort_values(['trip_id', 'stop_sequence'])
    filtered['stop_increment'] = np.random.uniform(-0.05, 0.12, size=n_rows)
    filtered['cumulative_increment'] = filtered.groupby('trip_id')['stop_increment'].cumsum()

    # 5. Final delay = combined delay + propagation + random noise
    filtered['delay_minutes'] = (
        filtered['pre_prop_delay'] +
        filtered['cumulative_increment'] +
        np.random.normal(0, 0.3, size=n_rows)
    ).clip(-3, 30).round(1)

    # --- Select final output columns ---
    output_cols = [
        'date', 'trip_id', 'route_id', 'direction_id', 'trip_headsign',
        'stop_id', 'stop_sequence', 'arrival_time', 'departure_time',
        'city', 'day_of_week', 'is_weekend', 'hour',
        'temp', 'precip', 'preciptype', 'conditions', 'windspeed',
        'humidity', 'cloudcover',
        'delay_minutes'
    ]
    return filtered[output_cols].copy()

# ===============================================================
# STEP 2: Process all 21 days
# Each day is processed independently and saved to a temp file
# to keep memory usage under control (8GB RAM constraint)
# ===============================================================
print(f"\nStep 2: Processing all {NUM_DAYS} days...\n")

all_parts = []
for day in range(1, NUM_DAYS + 1):
    current_date = START_DATE + timedelta(days=day - 1)
    day_name = current_date.strftime('%A %d %b %Y')
    print(f"  Processing day {day}/{NUM_DAYS} ({day_name})...", end=" ")

    result = process_one_day(day)

    if result is not None:
        mean_delay = result['delay_minutes'].mean()
        print(f"{result.shape[0]:,} rows | mean delay: {mean_delay:.2f} min")
        # Save to temp file to avoid holding all days in memory
        temp_file = f"temp_day_{day}.parquet"
        result.to_parquet(temp_file, index=False)
        all_parts.append(temp_file)
        del result

    gc.collect()

# ===============================================================
# STEP 3: Combine all days into one final parquet file
# ===============================================================
print(f"\nStep 3: Combining {len(all_parts)} daily files into final output...")

combined_parts = []
for f in all_parts:
    combined_parts.append(pd.read_parquet(f))

final = pd.concat(combined_parts, ignore_index=True)
del combined_parts
gc.collect()

# Save final dataset
final.to_parquet(OUTPUT_FILE, index=False)

# Clean up temporary files
for f in all_parts:
    os.remove(f)

print(f"\nFinal dataset saved: {OUTPUT_FILE}")
print(f"Done!")

---

## Stage 8: Final Dataset Summary & Verification

Load the final parquet file and verify the dataset characteristics.

In [ ]:
import pandas as pd
import os

final = pd.read_parquet("scotland_bus_delays_synthetic.parquet")

print("=" * 60)
print("  FINAL SYNTHETIC DATASET SUMMARY")
print("=" * 60)
print(f"Total rows: {final.shape[0]:,}")
print(f"Columns: {list(final.columns)}")
print(f"Date range: {final['date'].min()} to {final['date'].max()}")
print(f"Unique trips: {final['trip_id'].nunique():,}")
print(f"Unique routes: {final['route_id'].nunique()}")
print(f"Unique stops: {final['stop_id'].nunique():,}")

print(f"\n--- Delay Distribution ---")
print(f"Mean: {final['delay_minutes'].mean():.2f} min")
print(f"Median: {final['delay_minutes'].median():.2f} min")
print(f"Std: {final['delay_minutes'].std():.2f} min")
d_min = final['delay_minutes'].min()
d_max = final['delay_minutes'].max()
print(f"Min: {d_min:.1f}, Max: {d_max:.1f}")
early = (final['delay_minutes'] < 0).mean() * 100
ontime = ((final['delay_minutes'] >= -1) & (final['delay_minutes'] <= 3)).mean() * 100
late = (final['delay_minutes'] > 5).mean() * 100
vlate = (final['delay_minutes'] > 10).mean() * 100
print(f"% early (<0 min): {early:.1f}%")
print(f"% on time (-1 to 3 min): {ontime:.1f}%")
print(f"% late (>5 min): {late:.1f}%")
print(f"% very late (>10 min): {vlate:.1f}%")

print(f"\n--- By City ---")
print(final.groupby('city')['delay_minutes'].agg(['mean','median','count']).round(2).to_string())

print(f"\n--- By Day of Week ---")
dow_names = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}
dow_stats = final.groupby('day_of_week')['delay_minutes'].agg(['mean','count']).round(2)
for d in sorted(dow_stats.index):
    name = dow_names[d]
    mean_val = dow_stats.loc[d, 'mean']
    count_val = dow_stats.loc[d, 'count']
    print(f"  {name}: {mean_val:.2f} min ({count_val:,.0f} rows)")

print(f"\n--- By Weather Condition ---")
cond_stats = final.groupby('conditions')['delay_minutes'].agg(['mean','count']).round(2)
cond_stats = cond_stats.sort_values('mean', ascending=False)
for c in cond_stats.index:
    mean_val = cond_stats.loc[c, 'mean']
    count_val = cond_stats.loc[c, 'count']
    print(f"  {c}: {mean_val:.2f} min ({count_val:,.0f} rows)")

print(f"\n--- By Hour of Day ---")
hourly = final.groupby('hour')['delay_minutes'].mean()
for h in sorted(hourly.index):
    if pd.notna(hourly[h]):
        bar = '#' * max(0, int(hourly[h] * 3))
        print(f"  {h:02d}:00  {hourly[h]:5.2f} min  {bar}")

file_mb = os.path.getsize('scotland_bus_delays_synthetic.parquet') / 1e6
print(f"\nParquet file size: {file_mb:.1f} MB")

print(f"\n--- Sample Rows ---")
final.sample(5, random_state=42)

---

## Summary

### What Was Done:
1. **Inspected** the timetable parquet files — found ~8.97M rows per day covering all UK
2. **Filtered** to Edinburgh (620), Glasgow (609), Paisley (608) using NaPTAN stop_id prefixes
3. **Selected** the top 32 routes (15 Edinburgh + 15 Glasgow + 5 Paisley) by trip frequency
4. **Inspected** weather data — hourly, 5 regions, mapped Central→Glasgow/Paisley, East→Edinburgh
5. **Merged** timetable with weather by hour and region
6. **Synthesized** realistic delays calibrated from published transport research
7. **Processed** all 21 days and combined into a single parquet file

### Why Synthetic Delays:
Real-time arrival data for Scotland's Central Belt is not available through BODS.
The GTFS-RT feed was inspected and found to contain only VehiclePositions (GPS),
not TripUpdates (delays). Only ~10-34 vehicles appeared in the Central Belt area,
with most lacking trip_id fields. Scottish operators use separate systems outside BODS.

### Final Dataset:
- ~12.4 million rows across 21 days
- 32 routes, ~43,700 unique trips, ~2,067 stops
- Target variable: `delay_minutes` (realistic distribution: mean ~2.8 min)
- Weather features: temp, precip, preciptype, conditions, windspeed, humidity, cloudcover
- Time features: date, day_of_week, is_weekend, hour
- Route features: route_id, direction_id, stop_id, stop_sequence, city

### Stage 9: Add Engineered Feature - is_rush_hour

In [ ]:
import pandas as pd

final = pd.read_parquet("scotland_bus_delays_synthetic.parquet")

# Add is_rush_hour: 1 if hour is 7-9 (AM rush) or 16-18 (PM rush), 0 otherwise
final['is_rush_hour'] = final['hour'].apply(lambda h: 1 if h in [7, 8, 9, 16, 17, 18] else 0)

# Save updated file
final.to_parquet("scotland_bus_delays_synthetic.parquet", index=False)

print("is_rush_hour added and file saved.")
print(f"Rush hour rows: {final['is_rush_hour'].sum():,} ({final['is_rush_hour'].mean()*100:.1f}%)")
print(f"Non-rush rows: {(final['is_rush_hour']==0).sum():,}")
print(f"\nColumns now: {list(final.columns)}")

## Stage 10: Learning Curve Analysis

Before training the final models, we determine the optimal sample size by training XGBoost on increasing data sizes (10K to 900K rows) and measuring performance on a fixed test set. This identifies the point of diminishing returns where adding more data no longer improves model accuracy. This justifies our chosen sample size for model training.

In [ ]:
%pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load and prepare data
final = pd.read_parquet("scotland_bus_delays_synthetic.parquet")
sample = final.sample(n=1_000_000, random_state=42).copy()
del final

sample_encoded = pd.get_dummies(sample, columns=['city', 'conditions', 'preciptype'], drop_first=True)
drop_cols = ['delay_minutes', 'date', 'trip_id', 'route_id', 'stop_id',
             'trip_headsign', 'arrival_time', 'departure_time']
X = sample_encoded.drop(columns=drop_cols)
y = sample_encoded['delay_minutes']

# Fixed test set
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=100_000, random_state=42)

# Train on increasing sizes
sizes = [10_000, 50_000, 100_000, 200_000, 500_000, 900_000]
print(f"{'Sample Size':>12} {'MAE':>8} {'R2':>8}")
print("-"*30)

for n in sizes:
    X_sub = X_train_full.iloc[:n]
    y_sub = y_train_full.iloc[:n]
    
    model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
    model.fit(X_sub, y_sub)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{n:>12,} {mae:>8.4f} {r2:>8.4f}")

In [ ]:
# Ensure all output is displayed in full without truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

## Stage 11: Model Training and Evaluation

Based on the learning curve analysis, 500K rows captures virtually all learnable signal in the dataset. We train three regression models to predict delay_minutes:
- Linear Regression: baseline model for comparison
- Random Forest: ensemble of 100 decision trees
- XGBoost: gradient boosted trees (200 estimators)

Categorical features (city, conditions, preciptype) are one-hot encoded. Data is split 80/20 for train/test. Models are evaluated using MAE, RMSE, and R2 score.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Step 1: Load data and take a random sample
# 12.4M rows exceeds 8GB RAM during training, so we take a
# random sample of 500K rows. This preserves the proportional
# distribution of all features across cities, days, hours, etc.
# ============================================================
print("Step 1: Loading and sampling data...")
final = pd.read_parquet("scotland_bus_delays_synthetic.parquet")
sample = final.sample(n=500_000, random_state=42).copy()
del final
print(f"Sample size: {sample.shape[0]:,} rows")

# ============================================================
# Step 2: Encode categorical features using one-hot encoding
# - city: Edinburgh, Glasgow, Paisley (3 values)
# - conditions: 12 weather condition types
# - preciptype: None, Rain, Snow (3 values)
# drop_first=True avoids multicollinearity for Linear Regression
# ============================================================
print("\nStep 2: Encoding categorical features...")
sample_encoded = pd.get_dummies(sample, columns=['city', 'conditions', 'preciptype'], drop_first=True)

# ============================================================
# Step 3: Define features (X) and target variable (y)
# We drop ID/metadata columns that are not predictive features
# ============================================================
drop_cols = ['delay_minutes', 'date', 'trip_id', 'route_id', 'stop_id',
             'trip_headsign', 'arrival_time', 'departure_time']
X = sample_encoded.drop(columns=drop_cols)
y = sample_encoded['delay_minutes']

print(f"Features ({X.shape[1]}): {list(X.columns)}")
print(f"Target: delay_minutes")

# ============================================================
# Step 4: Train/Test split (80% train, 20% test)
# ============================================================
print("\nStep 3: Splitting train/test (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]:,} rows")
print(f"Test: {X_test.shape[0]:,} rows")

# ============================================================
# Step 5: Train all three models and evaluate
# - Linear Regression: baseline model
# - Random Forest: ensemble of decision trees
# - XGBoost: gradient boosted trees (typically best performer)
# ============================================================
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=100, max_depth=15, random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1
    )
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    # Predictions on test set
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results[name] = {
        'MAE': mae, 'RMSE': rmse, 'R2': r2,
        'Time': train_time, 'model': model, 'y_pred': y_pred
    }
    print(f"  MAE:  {mae:.4f} min")
    print(f"  RMSE: {rmse:.4f} min")
    print(f"  R2:   {r2:.4f}")
    print(f"  Training time: {train_time:.1f} seconds")

# ============================================================
# Step 6: Model comparison summary
# ============================================================
print("\n" + "="*60)
print("  MODEL COMPARISON SUMMARY")
print("="*60)
print(f"{'Model':<22} {'MAE':>8} {'RMSE':>8} {'R2':>8} {'Time(s)':>8}")
print("-"*60)
for name, m in results.items():
    print(f"{name:<22} {m['MAE']:>8.4f} {m['RMSE']:>8.4f} {m['R2']:>8.4f} {m['Time']:>8.1f}")

# ============================================================
# Step 7: Feature importance (from best tree-based model)
# ============================================================
print("\n" + "="*60)
print("  FEATURE IMPORTANCE (XGBoost)")
print("="*60)
xgb_model = results['XGBoost']['model']
importance = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False)
for feat, imp in importance.head(15).items():
    bar = '#' * int(imp * 100)
    print(f"  {feat:<35} {imp:.4f}  {bar}")